# VoiceContext Agent — How it works & interactive demo

This notebook explains the agent architecture **step by step** and lets you run each piece individually.

**No audio hardware required** — we start in text-only mode so you can understand the loop before adding NeMo.

---
## The agent loop (4 steps)

```
Audio/Text
   ↓
[1. PERCEIVE]  VoiceContext → TurnContext (text + emotion + paralinguistics + intent)
   ↓
[2. THINK]     LLM receives TurnContext + history + system prompt → produces reasoning
   ↓
[3. DECIDE]    LLM chooses: call a tool OR generate text reply
   ↓
[4. ACT]       Execute tool / speak reply → append to history → loop back to step 1
```

The key insight: **VoiceContext collapses steps 1-2 into a single MCP tool call** (`get_turn_context`). The LLM never touches raw audio — it only sees the structured object.

---
## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, "../apps/demo/src")   # simple_agent.py lives in src/

from dotenv import load_dotenv
load_dotenv("../.env")

if not os.environ.get("ANTHROPIC_API_KEY"):
    print("⚠️  ANTHROPIC_API_KEY not set — add it to your .env file")
else:
    print("✓ API key found")

---
## 1. Perceive — text → TurnContext

This is what VoiceContext does for you. We use a stub here; the real engine uses NeMo + SpeechBrain.

In [ ]:
from simple_agent import perceive_from_text
import json

# Try different sentences and observe how the TurnContext changes
test_sentences = [
    "I'm really frustrated, nothing is working and I've been waiting for an hour!",
    "Uh, hi, um, I was just wondering... how do I reset my password?",
    "Thanks, everything's working great now!",
    "I hate this service, I want a refund immediately.",
]

for sentence in test_sentences:
    ctx = perceive_from_text(sentence)
    print(f"Input  : {sentence[:60]}..." if len(sentence) > 60 else f"Input  : {sentence}")
    print(f"Emotion: {ctx.emotion.dominant} (valence={ctx.emotion.valence:+.2f}, arousal={ctx.emotion.arousal:.2f})")
    print(f"Intent : {ctx.intent.name} (conf={ctx.intent.confidence:.2f})")
    print(f"Hesit. : {ctx.paralinguistics.hesitations}")
    print()

print("─" * 50)
print("\nFull TurnContext object:")
ctx = perceive_from_text("I'm really frustrated, nothing is working!")
print(ctx.to_prompt_str())

---
## 2. Think — how the LLM sees TurnContext

The LLM receives the TurnContext as a formatted user message. Here's exactly what it looks like:

In [ ]:
from simple_agent import SYSTEM_PROMPT, perceive_from_text

ctx = perceive_from_text("I've been waiting for 3 days and nobody responded to my ticket!")

print("=== SYSTEM PROMPT (what the agent is) ===")
print(SYSTEM_PROMPT)

print("\n=== USER MESSAGE (what the LLM receives per turn) ===")
user_msg = f"[Turn 1 — {ctx.turn_id}]\n{ctx.to_prompt_str()}"
print(user_msg)

---
## 3. Decide — tools the agent can call

In [ ]:
from simple_agent import TOOLS
import json

print(f"Agent has {len(TOOLS)} tools:\n")
for tool in TOOLS:
    print(f"  • {tool['name']}")
    print(f"    {tool['description']}")
    props = tool['input_schema'].get('properties', {})
    print(f"    Args: {', '.join(props.keys())}")
    print()

---
## 4. One full agent loop — step by step

Let's trace a complete turn manually to understand exactly what happens.

In [ ]:
import anthropic
import json
from simple_agent import SYSTEM_PROMPT, TOOLS, execute_tool, perceive_from_text

client = anthropic.Anthropic()

# ─── Step 1: Perceive ─────────────────────────────────────────────
user_speech = "I'm so angry, your service is completely broken and I need help NOW!"
ctx = perceive_from_text(user_speech)

print("[1] PERCEIVE")
print(f"  Text    : {ctx.transcription.text}")
print(f"  Emotion : {ctx.emotion.dominant} (valence={ctx.emotion.valence:+.2f})")
print(f"  Intent  : {ctx.intent.name}")

# ─── Step 2: Build message for LLM ───────────────────────────────
history = [
    {"role": "user", "content": f"[Turn 1 — {ctx.turn_id}]\n{ctx.to_prompt_str()}"}
]

print("\n[2] THINK — calling LLM...")
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=512,
    system=SYSTEM_PROMPT,
    tools=TOOLS,
    messages=history,
)

print(f"  Stop reason : {response.stop_reason}")
print(f"  Content blocks: {[b.type for b in response.content]}")

# ─── Step 3: Decide ───────────────────────────────────────────────
print("\n[3] DECIDE")
if response.stop_reason == "tool_use":
    for block in response.content:
        if block.type == "tool_use":
            print(f"  → Tool call: {block.name}")
            print(f"    Input: {json.dumps(block.input, indent=6)}")
else:
    for block in response.content:
        if hasattr(block, "text"):
            print(f"  → Text reply: {block.text}")

# ─── Step 4: Act ──────────────────────────────────────────────────
print("\n[4] ACT")
if response.stop_reason == "tool_use":
    tool_results = []
    for block in response.content:
        if block.type == "tool_use":
            result = execute_tool(block.name, block.input)
            print(f"  Tool result: {json.dumps(result, indent=4)}")
            tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": json.dumps(result)})

    # LLM gets tool result and generates final reply
    history.append({"role": "assistant", "content": response.content})
    history.append({"role": "user", "content": tool_results})
    final_response = client.messages.create(
        model="claude-sonnet-4-6", max_tokens=256,
        system=SYSTEM_PROMPT, tools=TOOLS, messages=history
    )
    for block in final_response.content:
        if hasattr(block, "text"):
            print(f"\n  Final reply: {block.text}")

print(f"\n  History now has {len(history)+1} messages")
print("  → Loop back to step 1 for next turn")

---
## 5. Full agent — multi-turn conversation

Now let's run multiple turns and see how memory accumulates.

In [ ]:
from simple_agent import VoiceAgent, perceive_from_text

agent = VoiceAgent()

# Simulate a multi-turn support conversation
conversation = [
    "Hi, I need help with my account.",
    "I can't log in, I keep getting an error.",
    "Ugh, I've been trying for 30 minutes, this is so frustrating!",
    "Finally! Can you send me the reset link by email?",
]

print("=" * 60)
print("  Multi-turn voice agent conversation")
print("=" * 60)

for i, utterance in enumerate(conversation, 1):
    print(f"\n[Turn {i}]")
    print(f"🎤  User   : {utterance}")

    ctx = perceive_from_text(utterance)
    print(f"   Context: {ctx.emotion.dominant} (valence={ctx.emotion.valence:+.2f}), intent={ctx.intent.name}")

    response = agent.process_turn(ctx)
    print(f"🤖  Agent  : {response}")

print(f"\nHistory size: {len(agent.history)} messages ({agent.turn_count} turns)")

---
## 6. The power of emotion context — A/B test

Same words, different emotional context → different agent behavior.

In [ ]:
from simple_agent import VoiceAgent, TurnContext, Transcription, EmotionResult, ParalinguisticFeatures, IntentResult
import uuid
from datetime import datetime, timezone

def make_ctx(text, emotion, valence, arousal, hesitations=0):
    return TurnContext(
        turn_id=str(uuid.uuid4())[:8],
        timestamp=datetime.now(timezone.utc).isoformat(),
        transcription=Transcription(text=text),
        emotion=EmotionResult(dominant=emotion, valence=valence, arousal=arousal),
        paralinguistics=ParalinguisticFeatures(hesitations=hesitations),
        intent=IntentResult(name="inquiry", confidence=0.8),
    )

SAME_TEXT = "I need help with my account."
variants = [
    ("Calm, neutral",    make_ctx(SAME_TEXT, "neutral",  0.0,  0.1, hesitations=0)),
    ("Confused, hesitant",make_ctx(SAME_TEXT, "neutral",  0.0,  0.2, hesitations=4)),
    ("Angry, distressed",make_ctx(SAME_TEXT, "anger",   -0.75, 0.8, hesitations=0)),
    ("Happy, energetic", make_ctx(SAME_TEXT, "joy",      0.7,  0.6, hesitations=0)),
]

print(f"Text: '{SAME_TEXT}'  — same words, 4 different emotional contexts")
print("=" * 60)

for label, ctx in variants:
    agent = VoiceAgent()  # fresh agent each time
    response = agent.process_turn(ctx)
    print(f"\n[{label}]")
    print(f"  → {response}")

---
## 7. Connect real NeMo STT

Once the NeMo model is working from the streaming test, replace `perceive_from_text` with the real pipeline:

In [ ]:
# ─── REAL NEMO INTEGRATION ─────────────────────────────────────────────
# Uncomment once NeMo is verified in 01_nemo_fastconformer_streaming.ipynb

# import os
# os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
#
# import torch
# import nemo.collections.asr as nemo_asr
# from nemo.collections.asr.parts.utils.streaming_utils import CacheAwareStreamingAudioBuffer
#
# device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
# stt_model = nemo_asr.models.EncDecRNNTBPEModel.from_pretrained(
#     "nvidia/nemotron-speech-streaming-en-0.6b"
# ).to(device).eval()
#
# # Test with a real audio file
# with torch.no_grad():
#     result = stt_model.transcribe(["test_audio/your_file.wav"])
# text = result[0].text
#
# ctx = perceive_from_text(text)   # STT done, emotion still from stub
# agent = VoiceAgent()
# response = agent.process_turn(ctx)
# print(f"Agent: {response}")
print("Uncomment the cells above once NeMo is verified")

---
## 8. Run the full interactive demo

Open a terminal and run:

```bash
conda activate voicecontext
cd voicecontext

# Text mode (no audio needed)
python apps/demo/simple_agent.py --text

# Live mic (requires NeMo + pyaudio)
export PYTORCH_ENABLE_MPS_FALLBACK=1
python apps/demo/simple_agent.py --mic

# Audio file
python apps/demo/simple_agent.py --audio scripts/test_audio/nemo_sample.wav
```

## What's next

1. **Verify NeMo works** → run `notebooks/01_nemo_fastconformer_streaming.ipynb`
2. **Add real emotion** → integrate SpeechBrain into `perceive_from_audio()`
3. **Package as MCP** → `packages/mcp-server` — agents call `get_turn_context` via MCP protocol
4. **Add TTS** → response text → speech synthesis → speaker
5. **Deploy** → `docker/docker-compose.yml` — expose MCP over HTTP/SSE

In [ ]:
import os, time
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

# ── Check prerequisites ────────────────────────────────────────────────────
try:
    import torch
    import nemo.collections.asr as nemo_asr
    NEMO_OK = True
except ImportError as e:
    NEMO_OK = False
    print(f"NeMo not installed: {e}")
    print("\nTo install NeMo (heavy — only needed for real audio):")
    print("  uv add nemo-toolkit[asr] torch torchvision torchaudio")
    print("  # or for the demo venv:")
    print("  uv add --group dev nemo-toolkit[asr]")

if NEMO_OK:
    device = torch.device("mps" if torch.backends.mps.is_available()
                          else "cuda" if torch.cuda.is_available()
                          else "cpu")
    print(f"Device: {device}")

    # ── Load model (downloads ~600 MB on first run, cached after) ─────────
    print("Loading nvidia/nemotron-speech-streaming-en-0.6b ...")
    t0 = time.perf_counter()
    model = nemo_asr.models.EncDecRNNTBPEModel.from_pretrained(
        "nvidia/nemotron-speech-streaming-en-0.6b"
    ).to(device).eval()
    print(f"  Loaded in {time.perf_counter()-t0:.1f}s")

    # ── Transcribe a real file if provided, else synthesise silence ───────
    AUDIO_FILE = "../apps/demo/test_audio/sample.wav"   # ← change this path

    if os.path.exists(AUDIO_FILE):
        print(f"\nTranscribing: {AUDIO_FILE}")
        with torch.no_grad():
            result = model.transcribe([AUDIO_FILE])
        text = result[0].text if hasattr(result[0], "text") else str(result[0])
        print(f"  Transcript : {text!r}")

        # Feed directly into the agent
        from simple_agent import VoiceAgent, perceive_from_text
        ctx = perceive_from_text(text)
        agent = VoiceAgent()
        reply = agent.process_turn(ctx)
        print(f"\n  Emotion    : {ctx.emotion.dominant} (valence={ctx.emotion.valence:+.2f})")
        print(f"  Agent reply: {reply}")
    else:
        print(f"\n  No audio file found at {AUDIO_FILE}")
        print("  → Put a 16 kHz mono WAV there, or change AUDIO_FILE above")
        print("  → Streaming mic mode: run  python ../apps/demo/src/simple_agent.py --mic")

    # ── Quick model info ──────────────────────────────────────────────────
    num_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"\nModel params : {num_params:.0f} M")
    print(f"Sample rate  : {model.cfg.preprocessor.sample_rate} Hz")


---
## 11. Audio model — NeMo STT test

Tests whether the NeMo FastConformer model is available. If not, it prints clear install instructions and falls back to a silent stub so the rest of the notebook keeps running.

In [ ]:
from simple_agent import perceive_from_text

# Covers: anger, joy, uncertainty, neutral, complaint, inquiry, statement
PROBES = [
    ("anger / complaint",    "This is terrible! I've been waiting for hours and nothing works!"),
    ("joy / statement",      "This is amazing, everything works perfectly, thank you so much!"),
    ("uncertain / inquiry",  "Uh, hi, um, I was wondering... could you help me? I'm not sure what to do."),
    ("neutral / inquiry",    "How do I reset my password?"),
    ("neutral / statement",  "I received the confirmation email."),
    ("mixed",                "I love the product but I hate the checkout process, it's broken."),
]

print(f"{'Label':<25} {'Emotion':<10} {'Valence':>8} {'Arousal':>8} {'Intent':<12} {'Hesit.':>6}")
print("─" * 76)

for label, text in PROBES:
    ctx = perceive_from_text(text)
    print(
        f"{label:<25} "
        f"{ctx.emotion.dominant:<10} "
        f"{ctx.emotion.valence:>+8.2f} "
        f"{ctx.emotion.arousal:>8.2f} "
        f"{ctx.intent.name:<12} "
        f"{ctx.paralinguistics.hesitations:>6}"
    )

print("\n─── How to wire in real SpeechBrain emotion ──────────────────────────────")
print("""
from speechbrain.inference.classifiers import EncoderClassifier
classifier = EncoderClassifier.from_hparams(
    source="speechbrain/emotion-recognition-wav2vec2-IEMOCAP"
)
# audio_path → wav tensor → classifier → label + score
# Then build EmotionResult(dominant=label, valence=..., arousal=..., confidence=score)
""")


---
## 10. Emotion & Intent — deep probe

Test the perception stub across a wider range of sentences. Replace `perceive_from_text` with real SpeechBrain/NeMo output once the audio pipeline is ready.

In [ ]:
import time
import anthropic
from simple_agent import SYSTEM_PROMPT, TOOLS, perceive_from_text

client = anthropic.Anthropic()

def call_with_cache(user_text: str, history: list, use_cache: bool) -> tuple[str, dict]:
    system = (
        [{"type": "text", "text": SYSTEM_PROMPT, "cache_control": {"type": "ephemeral"}}]
        if use_cache else SYSTEM_PROMPT
    )
    t0 = time.perf_counter()
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=256,
        system=system,
        tools=TOOLS,
        messages=history,
    )
    latency_ms = (time.perf_counter() - t0) * 1000
    usage = response.usage
    return latency_ms, usage

turns = [
    "Hi, I need help with my account.",
    "I can't log in, I keep getting an error.",
    "I'm really frustrated, nothing is working!",
]

print("Turn-by-turn caching stats (cache_control ON)\n")
print(f"{'Turn':<6} {'Latency':>10} {'Input':>8} {'Cached':>8} {'Output':>8}")
print("─" * 46)

history = []
for i, text in enumerate(turns, 1):
    ctx = perceive_from_text(text)
    user_msg = {"role": "user", "content": f"[Turn {i}]\n{ctx.to_prompt_str()}"}
    history.append(user_msg)

    latency_ms, usage = call_with_cache(text, history, use_cache=True)
    cached = getattr(usage, "cache_read_input_tokens", 0)
    print(f"{i:<6} {latency_ms:>9.0f}ms {usage.input_tokens:>8} {cached:>8} {usage.output_tokens:>8}")

    # Append a stub assistant reply so history grows
    history.append({"role": "assistant", "content": "Stub reply."})

print("\n  ↑ 'Cached' tokens should rise from turn 2 onwards (system prompt cached)")
print("  Cache tokens cost 10% of normal input token price.")


---
## 9. Prompt Caching — cut latency & cost on repeated turns

The system prompt is identical on every turn. Marking it with `cache_control` tells Claude to cache it server-side so subsequent turns pay ~10 % of the input token cost and are measurably faster.